# Pipeline Principal de Análisis

Este notebook ejecuta el pipeline completo de procesamiento y análisis de datos del experimento de personalidad implícita y política.

## Pasos del Pipeline

1. Construcción de bases de datos
2. Cálculo de índices ideológicos
3. Creación de variables de cambio (CO y CT)
4. Limpieza de datos y eliminación de outliers
5. Tests estadísticos Mann-Whitney
6. Modelos de ecuaciones estructurales (SEM)
7. Generación de visualizaciones

In [ ]:
# ============================================================
# IMPORTACIONES
# ============================================================

import sys
from pathlib import Path

# Agregar rutas de módulos al path.
Ruta_Base = Path.cwd().parent
sys.path.append(str(Ruta_Base / "Codigo" / "Utilidades"))
sys.path.append(str(Ruta_Base / "Codigo" / "Procesamiento"))
sys.path.append(str(Ruta_Base / "Codigo" / "Test_Estadisticos"))
sys.path.append(
    str(Ruta_Base / "Codigo" / "Modelado_Estadistico")
)
sys.path.append(str(Ruta_Base / "Codigo" / "Visualizacion"))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from Configuracion import *
from Funciones_Comunes import *

print("✓ Módulos importados correctamente.")

## Paso 1: Construcción de Bases de Datos

In [ ]:
from Construccion_Bases import (
    Combinar_Archivos_Generales,
    Combinar_Archivos_Ballotage,
    Procesar_Base_Completa
)

# Cargar y procesar datos de Generales.
Df_Generales_Crudo = Combinar_Archivos_Generales()
Df_Generales = Procesar_Base_Completa(Df_Generales_Crudo)

# Cargar y procesar datos de Ballotage.
Df_Ballotage_Crudo = Combinar_Archivos_Ballotage()
Df_Ballotage = Procesar_Base_Completa(Df_Ballotage_Crudo)

print(f"\nDatos Generales: {len(Df_Generales)} filas")
print(f"Datos Ballotage: {len(Df_Ballotage)} filas")

## Paso 2: Cálculo de Índices Ideológicos

In [ ]:
from Calculo_Indices import Calcular_Todos_Indices

# Calcular índices para ambas bases.
Df_Generales = Calcular_Todos_Indices(Df_Generales)
Df_Ballotage = Calcular_Todos_Indices(Df_Ballotage)

# Mostrar estadísticas descriptivas.
print("\nEstadísticas de Índices (Generales):")
print(Df_Generales[[
    'Indice_Progresismo',
    'Indice_Conservadurismo',
    'Indice_Positividad'
]].describe())

## Paso 3: Creación de Variables de Cambio

In [ ]:
from Calculo_Variables_Cambio import Calcular_Variables_Cambio

# Crear diccionario con ambas bases.
Diccionario_Dfs = {
    'Generales': Df_Generales,
    'Ballotage': Df_Ballotage
}

# Calcular variables CO y CT.
Diccionario_Dfs = Calcular_Variables_Cambio(Diccionario_Dfs)

# Actualizar variables.
Df_Generales = Diccionario_Dfs['Generales']
Df_Ballotage = Diccionario_Dfs['Ballotage']

print("✓ Variables de cambio creadas.")

## Paso 4: Limpieza de Datos y Eliminación de Outliers

In [ ]:
from Limpieza_Datos import Limpiar_Diccionario_Dataframes

# Limpiar ambos DataFrames (categorías + tiempos).
Diccionario_Dfs = Limpiar_Diccionario_Dataframes(
    Diccionario_Dfs,
    Filtrar_Categorias=True,
    Filtrar_Tiempos=True,
    Numero_Desviaciones=3
)

# Actualizar variables.
Df_Generales = Diccionario_Dfs['Generales']
Df_Ballotage = Diccionario_Dfs['Ballotage']

print("✓ Datos limpios y sin outliers.")

## Paso 5: Tests Estadísticos (Ejemplo)

In [ ]:
from Mann_Whitney import Ejecutar_Mann_Whitney

# Ejemplo: Comparar CO_Item_3_Izq entre Left_Wing y
# Right_Wing_Libertarian.
Resultado = Ejecutar_Mann_Whitney(
    Df_Generales,
    'CO_Item_3_Izq',
    ['Left_Wing', 'Progressivism'],
    ['Right_Wing_Libertarian']
)

print("\nResultado del test Mann-Whitney:")
print(f"P-valor: {Resultado['P_Valor']:.4f}")
print(
    f"Significativo: {Resultado['Significativo']}"
)

## Paso 6: Visualización (Ejemplo)

In [ ]:
# Ejemplo de visualización simple.
Fig, Ax = plt.subplots(figsize=(10, 6))

Df_Generales.boxplot(
    column='Indice_Progresismo',
    by='Categoria_PASO_2023',
    ax=Ax
)

Ax.set_title(
    'Índice de Progresismo por Categoría Ideológica'
)
Ax.set_xlabel('Categoría PASO 2023')
Ax.set_ylabel('Índice de Progresismo')

plt.tight_layout()
plt.show()